## Backstage.io

In [ ]:
%%bash
kubectl create namespace backstage-system

helm repo add bitnami https://charts.bitnami.com/bitnami
helm repo update

helm install backstage-postgresql bitnami/postgresql \
  --namespace backstage-system \
  --set auth.username=backstage \
  --set auth.password=backstage123 \
  --set auth.database=backstagedb


Backstage.io

In [ ]:
%%bash
cat <<EOF >app-config.extra.yaml
catalog:
  locations:
    - type: url
      target: https://raw.githubusercontent.com/backstage/backstage/master/packages/catalog-model/examples/apis/hello-world-api.yaml
EOF
kubectl create configmap backstage-extra-config \
  --from-file=app-config.extra.yaml=app-config.extra.yaml \
  -n backstage-system


In [ ]:
%%bash
cat <<EOF >backstage-values.yaml
image:
  repository: ghcr.io/janus-idp/backstage-showcase
  tag: latest

appConfig:
  app:
    title: Backstage on MicroK8s
    baseUrl: http://localhost:7007
  extraAppConfig:
    - filename: app-config.extra.yaml
      configMapRef: backstage-extra-config    
  backend:
    baseUrl: http://localhost:7007
    listen:
      port: 7007
    database:
      client: pg
      connection:
        host: backstage-postgresql.backstage-system.svc.cluster.local
        port: 5432
        user: backstage
        password: backstage123
        database: backstagedb

postgresql:
  enabled: false

ingress:
  enabled: true
EOF

In [ ]:
%%bash
cat <<EOF >backstage-values.yaml

postgresql:
  enabled: true

ingress:
  enabled: true
EOF

In [ ]:
%%bash
helm repo add backstage https://backstage.github.io/charts
helm repo update
helm upgrade -i my-backstage backstage/backstage \
  --namespace backstage-system \
  -f backstage-values.yaml


In [ ]:
%%bash
kubectl exec -n backstage-system deployment/my-backstage -- cat /app/app-config.yaml

In [ ]:
%%bash
helm uninstall my-backstage  --namespace backstage-system


In [ ]:
%%bash
helm uninstall backstage-postgresql  --namespace backstage-system